In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [29]:
from pathlib import Path
import pandas as pd
import os
import matplotlib.pyplot as plt
import argparse

from scipy import signal
from scipy.interpolate import interp1d

In [5]:

for dirname, _, filenames in os.walk('data/raw_gen'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

data/raw_gen\hr60lfhf0.5.csv
data/raw_gen\hr61lfhf0.5.csv
data/raw_gen\hr62lfhf0.5.csv


In [15]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [35]:
# ==============================
# 1. Folder path
# ==============================

folder_path = "data\\raw_gen"

results = []

# ==============================
# 2. Clean RR
# ==============================

def clean_rr(rr):
    rr_clean = []
    for i in range(1, len(rr)):
        if 300 < rr[i] < 2000 and abs(rr[i] - rr[i-1]) < 0.2 * rr[i-1]:
            rr_clean.append(rr[i])
    return np.array(rr_clean)

# ==============================
# 3. Time-domain
# ==============================

def time_domain_features(nn):

    mean_rr = np.mean(nn)
    hr = 60000 / mean_rr
    sdnn = np.std(nn, ddof=1)

    diff_nn = np.diff(nn)
    rmssd = np.sqrt(np.mean(diff_nn**2))

    nn50 = np.sum(np.abs(diff_nn) > 50)
    pnn50 = 100 * nn50 / len(diff_nn) if len(diff_nn) > 0 else 0

    return mean_rr, hr, sdnn, rmssd, pnn50

# ==============================
# 4. Frequency-domain
# ==============================

def frequency_domain_features(nn):

    if len(nn) < 10:
        return 0, 0, 0

    t = np.cumsum(nn) / 1000.0
    t = t - t[0]

    fs = 4  # chuẩn HRV guideline
    f_interp = interp1d(t, nn, kind='cubic', fill_value="extrapolate")
    t_new = np.arange(0, t[-1], 1/fs)
    nn_interp = f_interp(t_new)

    nn_interp = nn_interp - np.mean(nn_interp)

    f, psd = signal.welch(nn_interp, fs=fs, nperseg=min(256, len(nn_interp)))

    lf_mask = (f >= 0.04) & (f < 0.15)
    hf_mask = (f >= 0.15) & (f < 0.4)

    lf = np.trapezoid(psd[lf_mask], f[lf_mask])
    hf = np.trapezoid(psd[hf_mask], f[hf_mask])

    lf_hf = lf / hf if hf > 0 else 0

    return lf, hf, lf_hf

# ==============================
# 5. Loop toàn bộ file
# ==============================

for filename in os.listdir(folder_path):

    if filename.endswith(".csv"):

        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Lấy R-peaks (Peak = 3)
        r_peaks = df[df["Peak"] == 3]["Time"].values

        if len(r_peaks) < 2:
            continue

        # RR (ms)
        rr = np.diff(r_peaks) * 1000

        # Clean -> NN
        nn = clean_rr(rr)

        if len(nn) < 5:
            continue

        # Extract features
        mean_rr, hr, sdnn, rmssd, pnn50 = time_domain_features(nn)
        lf, hf, lf_hf = frequency_domain_features(nn)

        results.append({
            "File": filename,
            "Mean_RR(ms)": mean_rr,
            "HR(bpm)": hr,
            "SDNN(ms)": sdnn,
            "RMSSD(ms)": rmssd,
            "pNN50(%)": pnn50,
            "LF Power": lf,
            "HF Power": hf,
            "LF/HF Ratio": lf_hf
        })

# ==============================
# 6. Convert to DataFrame
# ==============================

features_df = pd.DataFrame(results)

print(features_df)

# Optional: save
features_df.to_csv("data\\hrv_features.csv", index=False)

                     File  Mean_RR(ms)    HR(bpm)   SDNN(ms)   RMSSD(ms)  \
0  hr60hrstd3lfhf1.25.csv   997.977941  60.121569  50.194448   52.952362   
1   hr60hrstd6lfhf1.3.csv   993.627516  60.384801  97.045941  114.615220   
2   hr60hrtsd4lfhf1.3.csv   995.863971  60.249192  66.539153   70.002958   
3         hr60lfhf1.0.csv   998.667279  60.080070  41.883638   46.070963   
4        hr60lfhf1.25.csv   998.667279  60.080070  41.883638   46.070963   
5                test.csv   995.879289  60.248266  66.665031   70.652152   
6               testt.csv   995.879289  60.248266  66.665031   70.652152   
7              testtt.csv   998.636642  60.081913  41.902724   44.183978   

    pNN50(%)     LF Power     HF Power  LF/HF Ratio  
0  33.070866  1330.637337   767.831026     1.732982  
1  60.851064  4765.786223  3062.254041     1.556300  
2  45.669291  2407.617473  1319.246032     1.824995  
3  27.952756   831.308072   606.640563     1.370347  
4  27.952756   831.308072   606.640563     1.